# ORM 学习文档 — 从本项目代码出发

本文档基于项目实际代码，逐层讲解 ORM（对象关系映射）的使用方式。

## 什么是 ORM？

ORM = **O**bject **R**elational **M**apping（对象关系映射）。

它的核心思想是：**用 Python 类来表示数据库表，用类的实例来表示表中的一行数据**。

没有 ORM 时，你需要手写 SQL：
```sql
INSERT INTO user (name, password, email) VALUES ('张三', 'xxx', 'zhangsan@example.com');
SELECT * FROM user WHERE name = '张三';
```

有了 ORM 后，你用 Python 对象操作就行：
```python
user = User(name="张三", password="xxx", email="zhangsan@example.com")
session.add(user)
session.commit()

user = session.exec(select(User).where(User.name == "张三")).first()
```

**好处：**
- 不用拼接 SQL 字符串，减少 SQL 注入风险
- 用 Python 类型系统做校验，编译期就能发现类型错误
- 换数据库（比如 MySQL → PostgreSQL）只需要改连接字符串，业务代码不用动

## 本项目用的是什么 ORM？

本项目使用 **SQLModel**，它是 SQLAlchemy（Python 最成熟的 ORM）和 Pydantic（数据校验库）的结合体。

- `SQLModel, table=True` → 定义数据库表（同时具备 Pydantic 的数据校验能力）
- `BaseModel`（纯 Pydantic）→ 定义请求体 / 响应体，不涉及数据库

简单记忆：**需要建表就 `table=True`，不需要建表就 `BaseModel`**。

# 项目结构总览

```
app/
├── main.py                    # FastAPI app 实例、启动时建表、挂载路由
├── core/
│   ├── __init__.py
│   └── database.py            # 数据库引擎、连接池、session 生成器
├── models/
│   ├── __init__.py
│   ├── tb_user.py             # SQLModel 数据模型（ORM 表定义）
│   ├── request.py             # 请求体 Pydantic 模型
│   └── response.py            # 响应体 Pydantic 模型
├── routers/
│   ├── __init__.py
│   └── user.py                # 用户路由（接收 HTTP 请求 → 调用 service → 返回响应）
└── services/
    ├── __init__.py
    └── user.py                # 用户业务逻辑 + 密码工具 + 数据库操作
```

**分层职责：**

| 层 | 文件 | 职责 |
|---|---|---|
| **Router** | `routers/user.py` | 接收 HTTP 请求，调用 service，返回 HTTP 响应。不包含任何业务逻辑。 |
| **Service** | `services/user.py` | 所有业务逻辑和数据库操作。接收 session 作为参数。 |
| **Model** | `models/*.py` | 定义数据结构（数据库表、请求体、响应体）。 |

# 第一步：数据库配置 — `app/core/database.py`

这个文件负责三件事：
1. 创建数据库引擎（Engine）
2. 根据模型自动建表
3. 提供数据库会话（Session）给其他层使用

In [ ]:
# app/core/database.py — 完整代码 + 逐行注释

import os

from dotenv import load_dotenv     # 从 .env 文件读取环境变量
from sqlmodel import SQLModel, Session, create_engine

# --------------------------------------------------
# 1. 加载环境变量
# --------------------------------------------------
# load_dotenv() 会读取项目根目录的 .env 文件，
# 把里面的键值对注入到 os.environ 中。
# 这样数据库密码等敏感信息就不用硬编码在代码里。
# .env 文件内容示例：
#   DB_USER=root
#   DB_PASSWORD=123456
#   DB_HOST=localhost
#   DB_PORT=3306
#   DB_NAME=mydb
load_dotenv()

# --------------------------------------------------
# 2. 拼接数据库连接字符串
# --------------------------------------------------
# 格式：mysql+pymysql://用户名:密码@主机:端口/数据库名
#   mysql+pymysql  — 使用 pymysql 驱动连接 MySQL
#   如果用 PostgreSQL，则改成 postgresql://...
DATABASE_URL = (
    f"mysql+pymysql://{os.getenv('DB_USER')}:{os.getenv('DB_PASSWORD')}"
    f"@{os.getenv('DB_HOST')}:{os.getenv('DB_PORT', '3306')}/{os.getenv('DB_NAME')}"
)

# --------------------------------------------------
# 3. 创建数据库引擎（Engine）
# --------------------------------------------------
# Engine 是连接池的入口，管理着与数据库的连接。
# 注意：创建引擎时并没有真正连接数据库，只是准备好了配置。
# 真正的连接在第一次使用 session 时才建立（懒连接）。
engine = create_engine(
    DATABASE_URL,
    echo=True,              # 打印执行的 SQL 语句到终端，方便调试。生产环境建议设为 False。
    pool_size=10,           # 连接池中保持的常驻连接数
    max_overflow=20,        # 超出 pool_size 后允许创建的最大额外连接数
                            # 总连接数上限 = pool_size + max_overflow = 30
    pool_timeout=30,        # 连接池满了之后，获取连接的最大等待时间（秒），超时抛异常
    pool_recycle=900,       # 连接存活的最大时间（秒），超过后自动回收重建
                            # MySQL 默认 8 小时断开空闲连接，这里设 15 分钟主动回收，避免使用已断开的连接
    pool_pre_ping=True,     # 每次从池中取连接时先发一个 ping 检测是否存活
                            # 如果连接已断开，自动丢弃并创建新连接
)

# --------------------------------------------------
# 4. 自动建表
# --------------------------------------------------
# 读取所有继承自 SQLModel 且 table=True 的模型类，
# 执行 CREATE TABLE IF NOT EXISTS。
# 需要在 FastAPI 启动时调用一次（在 main.py 的 lifespan 里调用）。
# 注意：它不会修改已有的表结构（不会自动 ALTER TABLE）。
def create_db_and_tables():
    SQLModel.metadata.create_all(engine)


# --------------------------------------------------
# 5. 提供 Session 的生成器函数（用于 FastAPI 依赖注入）
# --------------------------------------------------
# Session 是你和数据库交互的"会话"。所有增删改查都通过 session 完成。
# with Session(engine) 确保用完后自动关闭连接，归还给连接池。
# yield 让它成为生成器，配合 FastAPI 的 Depends(get_session) 使用：
#   - 请求进来时，创建 session
#   - 请求处理完，自动关闭 session
def get_session():
    with Session(engine) as session:
        yield session

### 关键概念：Engine vs Session

| 概念 | 类比 | 作用 |
|---|---|---|
| **Engine** | 数据库的"总机" | 管理连接池，负责创建和回收连接。全局只创建一次。 |
| **Session** | 一次"通话" | 一次数据库交互的上下文。用完要关闭。每次 HTTP 请求创建一个。 |

# 第二步：定义数据模型

项目有三种模型，各司其职：

| 文件 | 用途 | 基类 | 是否建表 |
|---|---|---|---|
| `models/tb_user.py` | 数据库表模型（ORM） | `SQLModel, table=True` | 是 |
| `models/request.py` | 请求体校验 | `BaseModel`（纯 Pydantic） | 否 |
| `models/response.py` | 统一响应格式 | `BaseModel`（纯 Pydantic） | 否 |

三者的关系：
```
客户端发来 JSON 请求
    → Request 模型校验（字段类型对不对？必填字段有没有？）
    → User 模型（ORM）操作数据库
    → Response 模型格式化返回数据
    → 返回 JSON 响应给客户端
```

In [ ]:
# app/models/tb_user.py — 数据库表模型
#
# SQLModel 同时继承了 SQLAlchemy 的 ORM 能力和 Pydantic 的数据校验能力。
# table=True 是关键：加上它，SQLModel 才会为这个类创建数据库表。
# 不加 table=True 的 SQLModel 子类只是一个普通的数据容器。

from sqlmodel import SQLModel, Field


class User(SQLModel, table=True):       # table=True → 会创建一张名为 "user" 的表
    # Field() 是 SQLModel 提供的字段配置函数，可以指定主键、默认值、唯一约束等。
    #
    # id 字段：
    #   - int | None：创建时 id 为 None（数据库自动生成），读取时为 int
    #   - default=None：创建对象时如果不传 id，默认为 None
    #   - primary_key=True：标记为主键，数据库自动自增
    id: int | None = Field(default=None, primary_key=True)

    # name 字段：
    #   - str 类型，没有 default，意味着创建时必须提供
    #   - 没有设置 unique=True，允许重名（但业务层做了重名校验）
    name: str

    # password 字段：
    #   - 存储的是 bcrypt 哈希后的密码，不是明文
    #   - 数据库里看到的是类似 $2b$12$xxxxx... 的字符串
    password: str

    # email 字段：
    #   - unique=True 表示在数据库层强制邮箱唯一
    #   - 尝试插入重复邮箱会抛出数据库异常
    email: str = Field(unique=True)

In [ ]:
# app/models/request.py — 请求体模型
#
# 这些模型继承自 pydantic.BaseModel（不是 SQLModel），
# 所以不会创建数据库表，只用于 FastAPI 自动校验客户端发来的 JSON。
#
# FastAPI 会根据这些模型：
#   1. 自动校验请求体的字段类型
#   2. 自动生成 OpenAPI 文档（/docs 页面）
#   3. 字段缺失或类型错误时自动返回 422 错误

from pydantic import BaseModel


# 注册请求：客户端必须提供 name、password、email 三个字段
class UserCreateRequest(BaseModel):
    name: str
    password: str
    email: str


# 登录请求：用邮箱 + 密码登录
class UserLoginRequest(BaseModel):
    email: str
    password: str


# 修改密码请求
class PasswordUpdateRequest(BaseModel):
    id: int
    old_password: str
    new_password: str


# 修改用户名请求
class UserUpdateRequest(BaseModel):
    id: int
    name: str

In [ ]:
# app/models/response.py — 响应体模型
#
# 同样继承自 BaseModel，不涉及数据库。
# 用于统一 API 的返回格式，让前端知道响应的结构。

from typing import Any        # Any 表示可以是任意类型
from pydantic import BaseModel


# 所有接口统一使用这个格式返回
# code=0 表示成功，非 0 表示各种错误
class APIResponse(BaseModel):
    code: int = 0                    # 默认 0（成功）
    message: str = "success"         # 默认 "success"
    data: Any = None                 # 默认 None，可以是任意类型的数据


# 返回用户信息时，只暴露 name 和 email，不暴露 password
# 这样敏感字段就不会被返回给前端
class UserResponse(BaseModel):
    name: str
    email: str

# 第三步：Service 层 — `app/services/user.py`

这是整个项目的核心。业务逻辑和数据库操作全部集中在这里。

**设计原则：** 所有 service 函数都以 `session: Session` 作为第一个参数。session 由 router 层通过 `Depends(get_session)` 注入进来。这样 service 层不关心 session 怎么创建的，只管用它操作数据库。

In [ ]:
# app/services/user.py — 密码工具 + 业务逻辑

import bcrypt                        # 直接使用 bcrypt 库做密码哈希
from sqlmodel import Session, select  # Session：数据库会话，select：构建查询

from app.models.tb_user import User
from app.models.response import APIResponse, UserResponse


# --------------------------------------------------
# 密码工具
# --------------------------------------------------
# 永远不要在数据库中存储明文密码！
# bcrypt 是一种安全的密码哈希算法，特点：
#   - 同一个密码每次哈希的结果都不同（因为有随机盐值）
#   - 无法从哈希值反推出原始密码（单向函数）
#   - 计算速度故意很慢，增加暴力破解的成本

def hash_password(password: str) -> str:
    """将明文密码哈希，返回哈希字符串。"""
    # .encode("utf-8") 把字符串转成 bytes（bcrypt 需要 bytes 输入）
    # bcrypt.gensalt() 生成随机盐值
    # .decode("utf-8") 把哈希结果从 bytes 转回字符串，方便存入数据库
    return bcrypt.hashpw(password.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")


def verify_password(plain_password: str, hashed_password: str) -> bool:
    """验证密码是否匹配。返回 True/False。"""
    return bcrypt.checkpw(plain_password.encode("utf-8"), hashed_password.encode("utf-8"))

### CRUD 操作详解

CRUD = **C**reate / **R**ead / **U**pdate / **D**elete，是数据库的四种基本操作。

下面逐个讲解项目中每个操作对应的 ORM 用法。

In [ ]:
# --------------------------------------------------
# Create（创建）— 注册新用户
# --------------------------------------------------
#
# ORM 创建数据的核心步骤：
#   1. 创建模型实例（Python 对象）
#   2. session.add(obj)  — 把对象加入会话
#   3. session.commit()  — 提交事务，真正写入数据库
#   4. session.refresh(obj) — 刷新对象，拿到数据库生成的字段（如自增 id）
#
# 你可以把 session 想象成一个"暂存区"：
#   - add 只是把对象放进了暂存区，数据库里还没有
#   - commit 才是真正执行 SQL 把数据写入数据库
#   - refresh 是从数据库重新读取这一行，把数据库生成的字段回填到对象上

def register(session: Session, name: str, password: str, email: str) -> APIResponse:
    # 第一步：检查用户名是否已存在（先查再插，避免重复）
    # select(User) 相当于 SQL 的 "SELECT * FROM user"
    # .where(User.name == name) 相当于 "WHERE name = ?"
    # .first() 取第一条结果，没有则返回 None
    statement = select(User).where(User.name == name)
    existing = session.exec(statement).first()
    if existing is not None:
        return APIResponse(code=1, message="username already exists")

    # 第二步：创建模型实例
    # 注意：这里的 password 传的是哈希值，不是明文
    user = User(
        name=name,
        password=hash_password(password),
        email=email,
    )

    # 第三步：写入数据库
    session.add(user)       # 加入会话（暂存区）
    session.commit()        # 提交事务 → 执行 INSERT INTO user ...
    session.refresh(user)   # 刷新 → 从数据库重新读取，拿到数据库生成的 id

    # 第四步：返回响应（用 UserResponse 脱敏，不返回 password）
    return APIResponse(
        data=UserResponse(name=user.name, email=user.email),
    )

In [ ]:
# --------------------------------------------------
# Read（读取）— 登录 & 查询用户
# --------------------------------------------------
#
# ORM 查询数据有两种主要方式：
#
# 方式一：按主键查询
#   user = session.get(User, 1)
#   等价于 SQL: SELECT * FROM user WHERE id = 1
#   适合通过 id 查单条记录，最简单直接。
#
# 方式二：条件查询
#   statement = select(User).where(User.email == "xxx")
#   user = session.exec(statement).first()      # 取第一条
#   users = session.exec(statement).all()        # 取全部
#   等价于 SQL: SELECT * FROM user WHERE email = 'xxx'
#   适合按非主键字段查询，或者需要多条件组合的场景。

def login(session: Session, email: str, password: str) -> APIResponse:
    # 条件查询：按邮箱查找用户
    statement = select(User).where(User.email == email)
    user = session.exec(statement).first()
    if user is None:
        return APIResponse(code=1, message="user not found")
    # 验证密码：将明文密码与数据库中存储的哈希值比对
    if not verify_password(password, user.password):
        return APIResponse(code=2, message="wrong password")
    return APIResponse(
        data=UserResponse(name=user.name, email=user.email),
    )


def read_user(session: Session, user_id: int) -> APIResponse:
    # 主键查询：直接按 id 取，最快捷的方式
    user = session.get(User, user_id)
    if user is None:
        return APIResponse(code=1919810, message="user not found")
    return APIResponse(
        data=UserResponse(name=user.name, email=user.email)
    )

In [ ]:
# --------------------------------------------------
# Update（更新）— 修改密码 & 修改用户名
# --------------------------------------------------
#
# ORM 更新数据的核心步骤：
#   1. 先查出要更新的对象（session.get 或 session.exec）
#   2. 直接修改对象的属性（user.password = xxx）
#   3. session.add(obj)  — 告诉 session 这个对象有变更
#   4. session.commit()  — 提交事务，执行 UPDATE
#   5.（可选）session.refresh(obj) — 如果需要拿更新后的值

def update_password(session: Session, user_id: int, old_password: str, new_password: str) -> APIResponse:
    # 第一步：查出用户
    user = session.get(User, user_id)
    if user is None:
        return APIResponse(code=1919810, message="user not found")
    # 第二步：验证旧密码
    if not verify_password(old_password, user.password):
        return APIResponse(code=2, message="wrong password")
    # 第三步：修改字段
    user.password = hash_password(new_password)
    # 第四步：提交更新
    session.add(user)       # 标记对象已修改
    session.commit()        # 执行 UPDATE user SET password=? WHERE id=?
    return APIResponse(message="password updated")


def update_name(session: Session, user_id: int, name: str) -> APIResponse:
    user = session.get(User, user_id)
    if user is None:
        return APIResponse(code=1919810, message="user not found")
    # 检查新名字是否已被占用
    statement = select(User).where(User.name == name)
    existing = session.exec(statement).first()
    if existing is not None:
        return APIResponse(code=1, message="username already exists")
    # 修改并提交
    user.name = name                    # 修改字段
    session.add(user)                   # 标记已修改
    session.commit()                    # 执行 UPDATE
    session.refresh(user)               # 刷新，拿到最新值
    return APIResponse(data=UserResponse(name=user.name, email=user.email))

In [ ]:
# --------------------------------------------------
# Delete（删除）— 删除用户
# --------------------------------------------------
#
# ORM 删除数据的核心步骤：
#   1. 先查出要删除的对象
#   2. session.delete(obj)  — 标记删除
#   3. session.commit()     — 提交事务，执行 DELETE

def delete_user(session: Session, user_id: int) -> APIResponse:
    user = session.get(User, user_id)
    if user is None:
        return APIResponse(code=1919810, message="user not found")
    session.delete(user)    # 标记删除
    session.commit()        # 执行 DELETE FROM user WHERE id=?
    return APIResponse(message="user deleted")

# 第四步：Router 层 — `app/routers/user.py`

路由函数只做三件事：**接收 HTTP 请求 → 调用 service → 返回 HTTP 响应**。

关键机制是 `Depends(get_session)`：
- FastAPI 在处理请求前自动调用 `get_session()` 创建一个数据库 session
- 把 session 作为参数传给路由函数
- 请求处理完后自动关闭 session

这样路由函数完全不用关心数据库连接的创建和关闭。

In [ ]:
# app/routers/user.py

from fastapi import APIRouter, Depends, status
from sqlmodel import Session

from app.core import get_session
from app.models.response import APIResponse
from app.models.request import UserCreateRequest, UserLoginRequest, PasswordUpdateRequest, UserUpdateRequest
from app.services import user as user_service

# APIRouter 类似于 Flask 的 Blueprint，用于把路由分组。
# prefix="/user" 表示这个文件里所有路由都以 /user 开头。
# tags=["user"] 用于 /docs 页面的分组显示。
router = APIRouter(prefix="/user", tags=["user"])


# 每个 @router.xxx 装饰器定义一个 HTTP 接口：
#   - .get / .post / .put / .delete 对应 HTTP 方法
#   - status_code 指定成功时的 HTTP 状态码
#   - response_model 指定响应体的格式（用于自动生成文档）
#
# session: Session = Depends(get_session) 是 FastAPI 的依赖注入：
#   FastAPI 自动调用 get_session()，把返回的 session 传进来。

@router.get("/{user_id}", status_code=status.HTTP_200_OK, response_model=APIResponse)
def read_user(user_id: int, session: Session = Depends(get_session)):
    return user_service.read_user(session, user_id)


@router.post("/register", status_code=status.HTTP_201_CREATED, response_model=APIResponse)
def register(req: UserCreateRequest, session: Session = Depends(get_session)):
    # req 是请求体，FastAPI 自动解析 JSON 并用 UserCreateRequest 校验
    return user_service.register(session, req.name, req.password, req.email)


@router.post("/login", status_code=status.HTTP_200_OK, response_model=APIResponse)
def login(req: UserLoginRequest, session: Session = Depends(get_session)):
    return user_service.login(session, req.email, req.password)


@router.put("/password", status_code=status.HTTP_200_OK, response_model=APIResponse)
def update_password(req: PasswordUpdateRequest, session: Session = Depends(get_session)):
    return user_service.update_password(session, req.id, req.old_password, req.new_password)


@router.put("/name", status_code=status.HTTP_200_OK, response_model=APIResponse)
def update_name(req: UserUpdateRequest, session: Session = Depends(get_session)):
    return user_service.update_name(session, req.id, req.name)


@router.delete("/{user_id}", status_code=status.HTTP_200_OK, response_model=APIResponse)
def delete_user(user_id: int, session: Session = Depends(get_session)):
    return user_service.delete_user(session, user_id)

# 第五步：App 入口 — `app/main.py`

这是整个应用的入口，只做三件事：
1. **lifespan** — 应用启动时调用 `create_db_and_tables()` 自动建表
2. 创建 FastAPI 实例
3. 挂载路由（`include_router`）

In [ ]:
# app/main.py

from contextlib import asynccontextmanager

from fastapi import FastAPI, status

from app.core import create_db_and_tables
from app.models.response import APIResponse
from app.routers import user as user_router


# lifespan 是 FastAPI 的生命周期管理：
#   yield 之前 = 应用启动时执行
#   yield 之后 = 应用关闭时执行
@asynccontextmanager
async def lifespan(app):
    create_db_and_tables()   # 启动时：自动建表
    yield                    # 应用运行中...
    # （这里可以放关闭时的清理逻辑，本项目不需要）


app = FastAPI(lifespan=lifespan)

# 把 user_router 里定义的所有路由注册到 app 上
# 这样 /user/register、/user/login 等路径才能被访问到
app.include_router(user_router.router)


@app.get("/", status_code=status.HTTP_200_OK)
def read_root():
    return APIResponse(
        data={"content": "Hello, world!"},
    )


# 也可以直接用 uvicorn 运行：python -m app.main
if __name__ == "__main__":
    import uvicorn
    uvicorn.run(app, host="127.0.0.1", port=8000)

# 整体请求流程

以注册用户为例，一个完整的 HTTP 请求走过的路径：

```
1. 客户端发送 POST /user/register  {"name": "张三", "password": "123456", "email": "zs@example.com"}

2. FastAPI 接收请求
   ├── 路由匹配 → routers/user.py 的 register 函数
   ├── 请求体校验 → UserCreateRequest 自动校验字段类型和必填
   └── 依赖注入 → Depends(get_session) 自动创建数据库 session

3. Router 调用 Service
   → user_service.register(session, "张三", "123456", "zs@example.com")

4. Service 操作数据库（ORM）
   ├── select(User).where(User.name == "张三")  → 检查重名
   ├─│ User(name="张三", password="$2b$12$...", email="zs@example.com")  → 创建对象
   ├─│ session.add(user)   → 加入暂存区
   ├─│ session.commit()    → 执行 INSERT INTO user ...
   └─│ session.refresh()   → 拿到数据库生成的 id
       └── 返回 APIResponse(data=UserResponse(...))

5. Router 把 APIResponse 转成 JSON 返回给客户端
   → {"code": 0, "message": "success", "data": {"name": "张三", "email": "zs@example.com"}}

6. 请求结束，session 自动关闭，连接归还给连接池
```

# ORM 操作速查表

## CRUD 基本操作

| 操作 | ORM 代码 | 对应 SQL |
|---|---|---|
| **新增** | `session.add(obj)` + `session.commit()` + `session.refresh(obj)` | `INSERT INTO ...` |
| **主键查询** | `session.get(Model, 主键)` | `SELECT * FROM ... WHERE id = ?` |
| **条件查询（单条）** | `session.exec(select(Model).where(...)).first()` | `SELECT * FROM ... WHERE ... LIMIT 1` |
| **条件查询（全部）** | `session.exec(select(Model).where(...)).all()` | `SELECT * FROM ... WHERE ...` |
| **查询所有** | `session.exec(select(Model)).all()` | `SELECT * FROM ...` |
| **更新** | 修改属性 → `session.add(obj)` + `session.commit()` | `UPDATE ... SET ... WHERE id = ?` |
| **删除** | `session.delete(obj)` + `session.commit()` | `DELETE FROM ... WHERE id = ?` |

## 查询条件组合

```python
from sqlmodel import select, col

# 等于
select(User).where(User.name == "张三")

# 不等于
select(User).where(User.name != "张三")

# 多个条件（AND）
select(User).where(User.name == "张三", User.email.like("%@example.com"))

# 排序
select(User).order_by(User.id)
select(User).order_by(col(User.id).desc())   # 倒序

# 限制数量
select(User).limit(10).offset(20)   # 分页：跳过前 20 条，取 10 条
```

## 事务

```python
# session.commit() 提交当前事务
# session.rollback() 回滚当前事务（撤销未提交的修改）
# 每次 commit 后会自动开启一个新事务
#
# with Session(engine) 自动管理：
#   - 正常退出 → commit
#   - 异常退出 → rollback
```

## 常见陷阱

| 陷阱 | 说明 |
|---|---|
| 忘记 `commit()` | `add` / `delete` 只是暂存，不 `commit` 不会真正写入数据库 |
| 忘记 `refresh()` | 新增后如果不 `refresh`，对象的 `id` 仍然是 `None` |
| 修改属性后忘记 `add` | 直接改属性后必须 `session.add(obj)` 再 `commit`，否则 ORM 不知道对象被修改了 |
| 查询结果为 None | `session.get()` 和 `.first()` 可能返回 `None`，使用前一定要做判空 |